In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')
LOG_DIR = "/content/drive/MyDrive/Research_Logs"

print(f"✅ ログ保存先: {LOG_DIR}")

%pip install rank_bm25
%pip install xopen

from pathlib import Path
import sys
import torch
from datasets import load_dataset
from datetime import datetime
import xopen
import random

import huggingface_hub
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
huggingface_hub.login(hf_token)

random.seed(0)


# 以下自作モジュール
module_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/my_modules"
if module_path not in sys.path:
    sys.path.append(module_path)

def confirmation():
    """実行前にユーザーの確認を求める"""
    print("\n" + "!" * 30)
    print("警告: 書き出しモードが 'a' (追記) です。")
    print("既存のファイルにデータが追加されますが、よろしいですか？")
    print("!" * 30 + "\n")

    answer = input("実行する場合は 'yes' と入力してください: ").strip().lower()

    if answer != 'yes':
        print("\n[中止] 実行がキャンセルされました。")
        # Jupyterでセルを止める最もクリーンな方法
        raise KeyboardInterrupt("User cancelled the execution.")

    print("\n[開始] 実験を実行します...\n")

Mounted at /content/drive
✅ ログ保存先: /content/drive/MyDrive/Research_Logs
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.4/132.4 kB 18.4 MB/s eta 0:00:00


In [2]:
import torch
import sys
from transformers import AutoTokenizer, AutoModelForCausalLM
from DynamicScalingLlamaAttention_last_attnweight import DynamicScalingLlamaAttention
CACHED_MODEL = None
CACHED_TOKENIZER = None

def get_model_safe(model_path):
    global CACHED_MODEL, CACHED_TOKENIZER

    if CACHED_MODEL is not None:
        print("✅ Model already loaded. Using cached model.")
        return CACHED_MODEL, CACHED_TOKENIZER

    print("⏳ Loading new model... (This may take a while)")

    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        attn_implementation="eager",
        dtype=torch.float16,
        use_cache=False,
        trust_remote_code=True
    )

    if len(model.model.layers) > 20:
        print(f"✂️ Truncating model layers from {len(model.model.layers)} to 20")
        model.model.layers = model.model.layers[:20]

    gc.collect()
    torch.cuda.empty_cache()

    print("🔄 Swapping Attention Modules to DynamicScalingLlamaAttention...")

    target_layers = list(range(14,20))

    # モデル設定
    model.config.hidden_scale_config = {
        "target_layers": target_layers,
        "target_dims": [2393],
        "factor": 1,
        "last_recompute_tokens": 1,
        "change_value": False
    }

    # 全レイヤー入れ替え
    for i, layer in enumerate(model.model.layers):
        if i in target_layers:
            original_attn = layer.self_attn
            target_device = original_attn.q_proj.weight.device
            target_dtype = original_attn.q_proj.weight.dtype

            new_attn = DynamicScalingLlamaAttention(config=model.config, layer_idx=i)

            # originalの情報をコピー
            new_attn.load_state_dict(original_attn.state_dict(), strict=False)
            new_attn.to(device=target_device, dtype=target_dtype)
            layer.self_attn = new_attn
        else: pass

    CACHED_MODEL = model
    CACHED_TOKENIZER = tokenizer

    print("✅ Model loaded and patched successfully.")
    return CACHED_MODEL, CACHED_TOKENIZER

In [3]:
from datasets.formatting.formatting import TypeVar
from typing import List, Optional, Type
from pydantic.dataclasses import dataclass
from rank_bm25 import BM25Okapi
import numpy as np


PROMPTS_ROOT = Path("/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/lost_in_the_middle/prompts")
T = TypeVar("T")

@dataclass(frozen=True)
class Document:
    title: str
    text: str
    id: Optional[str] = None
    score: Optional[float] = None
    hasanswer: Optional[bool] = None
    isgold: Optional[bool] = None
    original_retrieval_index: Optional[int] = None

    @classmethod
    def from_dict(cls: Type[T], data: dict) -> T:
        data = deepcopy(data)
        if not data:
            raise ValueError("Must provide data for creation of Document from dict.")
        id = data.pop("id", None)
        score = data.pop("score", None)
        isgold = data.pop("isgold", None)
        # Convert score to float if it's provided.
        if score is not None:
            score = float(score)
        return cls(**dict(data, id=id, score=score, isgold=isgold))

def get_qa_prompt_reorder(question, documents, trial_orders):
    """
    questions: [q, q, ...] (バッチサイズ分のリスト)
    documents: [[Doc, Doc, ...], [Doc, ...], ...] (各試行用のドキュメントリストのリスト)
    trial_orders: [[2, 0, 1, ...], [5, 2, ...]] (ランダムな順序インデックスのリスト)
    """
    prompt_filename = "qa.prompt"

    with open(PROMPTS_ROOT / prompt_filename) as f:
        prompt_template = f.read().rstrip("\n")

    all_prompts = []
    all_doc_positions = []

    for trial_order in trial_orders:

        # --- 1. 指定されたランダム順序（trial_order）でドキュメントを並び替え ---
        # trial_order [2, 0, 1] なら、元のリストの2番目, 0番目, 1番目の順になる
        reordered_documents = [documents[i] for i in trial_order]

        # --- 2. ドキュメントのテキスト化 ---
        formatted_document_strings = []
        for document_index, doc_obj in enumerate(reordered_documents):
            # Documentクラスの属性(title, text)を使用してフォーマット
            doc_string = f"Document [{document_index+1}](Title: {doc_obj.title}) {doc_obj.text}"
            formatted_document_strings.append(doc_string)

        # --- 3. プロンプト全体の組み立て ---
        search_results_text = "\n\n".join(formatted_document_strings)
        prompt = prompt_template.format(question=question, search_results=search_results_text)

        # --- 4. 各ドキュメントの文字位置（start, end）の特定 ---
        doc_positions = []
        for doc_string in formatted_document_strings:
            start_idx = prompt.find(doc_string)

            if start_idx == -1:
                # デバッグ用：見つからない場合は警告
                print(f"Warning: Text not found in prompt: {doc_string[:30]}...")
                end_idx = -1
            else:
                end_idx = start_idx + len(doc_string)

            doc_positions.append({
                "start": start_idx,
                "end": end_idx
            })

        all_prompts.append(prompt)
        all_doc_positions.append(doc_positions)

    return all_prompts, all_doc_positions

def get_attnmap(question, documents, model, tokenizer, trial_orders):
    """各文書のトークンに対するアテンション重みの取得を行う（ランダム順序対応版）"""

    # プロンプトの作成と位置取得
    # trial_orders は [[2, 0, 1, ...], [0, 2, 1, ...]] のような「インデックスのリスト」のリストを想定
    all_prompts, all_doc_positions = get_qa_prompt_reorder(question, documents, trial_orders)

    # トークナイズ
    inputs = tokenizer(
        all_prompts,
        padding=True,
        return_tensors="pt",
        return_offsets_mapping=True,
    ).to(model.device)
    offset_mapping = inputs.pop("offset_mapping")

    # Hookの設定
    layer_attns = {}
    target_layer_indices = range(14, 20)
    def hook_fn(layer_idx):
        def fn(module, input, output):
            attn_weights = module.attn_weights_last
            if attn_weights is not None:
                layer_attns[layer_idx] = attn_weights.detach().cpu()
                module.attn_weights_last = None
        return fn

    hooks = []
    for i in target_layer_indices:
        h = model.model.layers[i].self_attn.register_forward_hook(hook_fn(i))
        hooks.append(h)

    # Prefill
    try:
        with torch.inference_mode():
            _ = model(**inputs)

        # 各文書の重みを集計
        # values()の順序を保証するためキーでソートしてstack
        sorted_layers = [layer_attns[i] for i in sorted(layer_attns.keys())]
        all_selected_attn = torch.stack(sorted_layers, dim=1) # [batch, layers, heads, 1, seq_len]
        avg_attn_matrix = all_selected_attn.mean(dim=3) # [batch, layers, heads, seq_len]

        batch_doc_weights = []
        for batch_count in range(len(avg_attn_matrix)):
            offsets = torch.tensor(offset_mapping[batch_count])
            s_base, e_base = offsets[:, 0], offsets[:, 1]

            # 各文書のアテンションの取得
            current_doc_weights = []
            for pos in all_doc_positions[batch_count]:
                start_char, end_char = pos["start"], pos["end"]
                mask = (s_base != e_base) & (e_base >= start_char) & (s_base <= end_char)
                doc_token_indices = torch.where(mask)[0].tolist()

                if not doc_token_indices:
                    raise ValueError("doc_token_indices not found")

                doc_attn_weights = avg_attn_matrix[batch_count, :, :, doc_token_indices] # (layers, heads, doc_len)
                doc_attn = doc_attn_weights.mean(dim=2) # (layers, heads)
                current_doc_weights.append(doc_attn)

            # --- 並び替え（復元）ロジック ---
            num_docs = len(documents)
            # trial_order は現在のバッチにおける並び順リスト（例: [2, 0, 1...]）
            trial_order = trial_orders[batch_count]

            # 安全策：長さが一致しているか確認
            if len(current_doc_weights) != len(trial_order):
                raise ValueError(f"Length mismatch: weights({len(current_doc_weights)}) vs trial_order({len(trial_order)})")

            original_ordered_scores = [None] * num_docs
            for j, score in enumerate(current_doc_weights):
                # プロンプトの j 番目にある文書の「元のインデックス」を trial_order から取得
                original_idx = trial_order[j]
                original_ordered_scores[original_idx] = score

            # ここで None が残っていないか最終確認
            if any(s is None for s in original_ordered_scores):
                missing_indices = [idx for idx, s in enumerate(original_ordered_scores) if s is None]
                raise ValueError(f"Failed to fill all scores. Missing original indices: {missing_indices}. trial_order: {trial_order}")
            batch_doc_weights.append(original_ordered_scores)

    finally:
        for h in hooks:
            h.remove()

    return batch_doc_weights

def get_bm25_indices(question, documents):
    """
    質問と文書群を受け取り、BM25スコアが高い順のインデックスリストを返す
    """
    # トークナイズ（簡易版）
    tokenized_corpus = [doc.text.lower().split() for doc in documents]
    bm25 = BM25Okapi(tokenized_corpus)

    tokenized_query = question.lower().split()
    # 文書ごとのスコアを取得
    doc_scores = bm25.get_scores(tokenized_query)

    # スコア降順のインデックスを返す
    # argsortは昇順なので [::-1] で反転させる
    return np.argsort(doc_scores)[::-1].tolist()

In [4]:
from xopen import xopen
from copy import deepcopy
from tqdm import tqdm
import json
import random
import os
import gc
import torch
import subprocess
import time

# --- 設定 (必要に応じて調整してください) ---
SYNC_INTERVAL = 100  # 100件ごとにDriveへ同期
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

class SafeDriveSaver:
    def __init__(self, target_drive_path: str):
        self.drive_path = target_drive_path
        # ローカルの一時ファイルパス (/content/ファイル名)
        self.local_path = os.path.join("/content", os.path.basename(target_drive_path))

        # 保存先ディレクトリ作成
        os.makedirs(os.path.dirname(self.drive_path), exist_ok=True)

        # 1. 初期ロード (Drive -> Local)
        if os.path.exists(self.drive_path):
            print(f"🔄 Driveから既存データをロード中: {self.drive_path}")
            # エラーチェック付きでコピー
            res = subprocess.run(f'cp "{self.drive_path}" "{self.local_path}"', shell=True)
            if res.returncode != 0:
                raise RuntimeError("❌ Driveからのデータ読み込みに失敗しました。停止します。")
        else:
            print(f"🆕 新規ファイルとして開始します")

    @property
    def temp_file_path(self) -> str:
        return self.local_path

    def get_current_count(self) -> int:
        """現在ローカルに保存されている行数を返す"""
        if not os.path.exists(self.local_path):
            return 0
        with open(self.local_path, "r", encoding="utf-8") as f:
            return sum(1 for _ in f)

    def save_to_local(self, file_handle, data_dict):
        """ローカル一時保存 (高速)"""
        file_handle.write(json.dumps(data_dict) + "\n")
        file_handle.flush()
        os.fsync(file_handle.fileno())

    def sync_to_drive(self):
        """Driveへ同期 (安全版: .bak作成 -> 上書き -> sync)"""
        if not os.path.exists(self.local_path):
            return

        # バックアップ作成
        if os.path.exists(self.drive_path):
            subprocess.run(f'cp -f "{self.drive_path}" "{self.drive_path}.bak"', shell=True)

        # メインの上書き同期
        subprocess.run(f'cp -f "{self.local_path}" "{self.drive_path}"', shell=True)
        subprocess.run('sync', shell=True)

# ==========================================
# ▼ メイン処理 (Random Permutation)
# ==========================================
def main():
    # 1. 保存管理クラスの初期化
    saver = SafeDriveSaver(OUTPUT_PATH)

    # 2. 再開位置の特定
    done_count = saver.get_current_count()
    print(f"Resume check: {done_count} lines already processed.")

    # モデル準備
    model, tokenizer = get_model_safe(MODEL_NAME)
    model.config.current_scale_map = None
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    gc.collect()
    torch.cuda.empty_cache()

    # 3. ファイルオープン (saver.temp_file_path を開く)
    with xopen(NQ_path, "r") as fin, open(saver.temp_file_path, "a", encoding="utf-8") as fout:

        # tqdmの設定 (total=DATA_NUM に修正済み)
        pbar = tqdm(fin, total=DATA_NUM)

        # セッション内での処理数カウント
        processed_in_session = 0

        for i, line in enumerate(pbar):
            # 再開スキップロジック
            if i < done_count + VALIDATION_SIZE:
                continue
            if i >= DATA_NUM:
                break

            random.seed(i)

            try:
                data = json.loads(line)
                question = data["question"]

                documents = []
                for ctx in deepcopy(data["ctxs"]):
                    documents.append(Document.from_dict(ctx))
                if not documents:
                    # 元コードの通りエラーを発生させる
                    raise ValueError(f"Did not find any documents: {data}")

                bm25_indices = get_bm25_indices(question, documents)
                bm25_reordered_documents = [documents[idx] for idx in bm25_indices]

                all_attnmaps = []

                # --- ランダム順序の生成 (ここがRandom版の特徴) ---
                # ドキュメント数20件と仮定して 0~19 のランダムな並びを TRIAL_NUM 個作る
                trial_orders_pool = [random.sample(range(20), 20) for _ in range(TRIAL_NUM)]

                # バッチ処理ループ
                for start_idx in range(0, TRIAL_NUM, BATCH_SIZE):
                    end_idx = min(start_idx + BATCH_SIZE, TRIAL_NUM)
                    trial_orders = trial_orders_pool[start_idx:end_idx]

                    attnmaps = get_attnmap(
                        question,
                        bm25_reordered_documents,
                        model, tokenizer, trial_orders
                    )

                    for trial_attnmap in attnmaps:
                        restored_attnmap = [None] * len(documents)
                        for j, score in enumerate(trial_attnmap):
                            original_idx = bm25_indices[j]
                            restored_attnmap[original_idx] = score.tolist()
                        all_attnmaps.append(restored_attnmap)

                    del attnmaps
                    torch.cuda.empty_cache()

                # 出力データ作成
                output_dict = {
                    "id": i,
                    "question": question,
                    "attnmap": all_attnmaps,
                }

                # --- 保存と同期 ---
                # 1. ローカルへ保存
                saver.save_to_local(fout, output_dict)

                # 2. 定期同期 (100件ごと)
                processed_in_session += 1
                if processed_in_session % SYNC_INTERVAL == 0:
                    pbar.set_description("Syncing to Drive...")
                    saver.sync_to_drive()
                    pbar.set_description("Processing")

            except Exception as e:
                print(f"Error at index {i}: {e}")
                saver.sync_to_drive() # エラー時も保存
                raise e

    # 終了時の最終同期
    print("Finalizing save...")
    saver.sync_to_drive()

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("Done.")

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

NQ_path = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_random_6trial_llama.jsonl"

TRIAL_NUM = 6
BATCH_SIZE = 3

VALIDATION_SIZE = 600
DATA_NUM = 2655

MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"


if __name__ == "__main__":
  main()

⏳ Loading new model... (This may take a while)


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

🔄 Swapping Attention Modules to DynamicScalingLlamaAttention...
✅ Model loaded and patched successfully.


2655it [00:01, 1697.33it/s]


In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

NQ_path = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_random_12trial_llama.jsonl"

TRIAL_NUM = 12
BATCH_SIZE = 3

VALIDATION_SIZE = 600
DATA_NUM = 2655

MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"


if __name__ == "__main__":
  main()

🔄 Driveから既存データをロード中: /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_random_12trial_llama.jsonl


KeyboardInterrupt: 

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

NQ_path = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_random_6trial_vicuna-16k.jsonl"

TRIAL_NUM = 6
BATCH_SIZE = 3

VALIDATION_SIZE = 600
DATA_NUM = 2655

MODEL_NAME = "lmsys/vicuna-7b-v1.5-16k"


if __name__ == "__main__":
  main()

⏳ Loading new model... (This may take a while)


config.json:   0%|          | 0.00/692 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/750 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🔄 Swapping Attention Modules to DynamicScalingLlamaAttention...
✅ Model loaded and patched successfully.


2655it [00:01, 1649.05it/s]


In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

NQ_path = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
OUTPUT_PATH = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_random_12trial_vicuna-16k.jsonl"

TRIAL_NUM = 12
BATCH_SIZE = 4

VALIDATION_SIZE = 600
DATA_NUM = 2655

MODEL_NAME = "lmsys/vicuna-7b-v1.5-16k"


if __name__ == "__main__":
  main()

🔄 Driveから既存データをロード中: /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_random_12trial_vicuna-16k.jsonl
Resume check: 1613 lines already processed.
⏳ Loading new model... (This may take a while)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✂️ Truncating model layers from 32 to 20
🔄 Swapping Attention Modules to DynamicScalingLlamaAttention...
✅ Model loaded and patched successfully.


  0%|          | 0/2655 [00:00<?, ?it/s]/tmp/ipython-input-1271202468.py:132: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  offsets = torch.tensor(offset_mapping[batch_count])
Processing: 100%|██████████| 2655/2655 [2:34:59<00:00,  3.50s/it]


Finalizing save...
Done.


In [ ]:
# 1. ローカルで作った修正済みファイルをDriveに強制コピー
!cp -f /content/repair_temp.jsonl "{OUTPUT_PATH}"

# 2. OSのバッファを強制フラッシュ
import subprocess
subprocess.run("sync", shell=True)

print("✅ Driveへの強制上書きが完了しました。")

# 3. 再確認
!wc -l "{OUTPUT_PATH}"

In [ ]:
!wc -l "/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_random_12trial_vicuna-16k.jsonl"

2055 /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_random_12trial_vicuna-16k.jsonl


In [5]:
from xopen import xopen
from copy import deepcopy
from tqdm import tqdm
import json
import random
import os
import gc
import itertools
import subprocess
import time

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- 設定 (必要に応じて調整してください) ---
SYNC_INTERVAL = 100  # 何件ごとにDriveへ同期するか

class SafeDriveSaver:
    def __init__(self, target_drive_path: str):
        self.drive_path = target_drive_path
        # ローカルの一時ファイルパス
        self.local_path = os.path.join("/content", os.path.basename(target_drive_path))

        # 保存先ディレクトリ作成
        os.makedirs(os.path.dirname(self.drive_path), exist_ok=True)

        # 1. 初期ロード (Drive -> Local)
        if os.path.exists(self.drive_path):
            print(f"🔄 Driveから既存データをロード中: {self.drive_path}")
            # エラーチェック付きでコピー
            res = subprocess.run(f'cp "{self.drive_path}" "{self.local_path}"', shell=True)
            if res.returncode != 0:
                raise RuntimeError("❌ Driveからのデータ読み込みに失敗しました。停止します。")
        else:
            print(f"🆕 新規ファイルとして開始します")

    @property
    def temp_file_path(self) -> str:
        return self.local_path

    def get_current_count(self) -> int:
        if not os.path.exists(self.local_path):
            return 0
        with open(self.local_path, "r", encoding="utf-8") as f:
            return sum(1 for _ in f)

    def save_to_local(self, file_handle, data_dict):
        """ローカル一時保存 (変更なし)"""
        file_handle.write(json.dumps(data_dict) + "\n")
        file_handle.flush()
        os.fsync(file_handle.fileno())

    def sync_to_drive(self):
        """
        【安全版】Driveへ同期
        1. 現在のDrive上のファイルを .bak にバックアップ
        2. ローカルのファイルをDriveへ上書き
        """
        if not os.path.exists(self.local_path):
            return

        # --- 追加: バックアップ作成 ---
        # Driveにファイルがある場合のみバックアップを作る
        if os.path.exists(self.drive_path):
            backup_cmd = f'cp -f "{self.drive_path}" "{self.drive_path}.bak"'
            subprocess.run(backup_cmd, shell=True)
        # ---------------------------

        # メインの上書き同期
        subprocess.run(f'cp -f "{self.local_path}" "{self.drive_path}"', shell=True)
        subprocess.run('sync', shell=True)

def generate_block_permuted_orders(num_docs=20, num_repeats=1):
    """(変更なし)"""
    indices = list(range(num_docs))
    block1 = indices[0:7]   # 0~6
    block2 = indices[7:13]  # 7~12
    block3 = indices[13:20] # 13~19

    blocks = [block1, block2, block3]
    block_perms = list(itertools.permutations(blocks))
    trial_orders = []

    for _ in range(num_repeats):
        for perm in block_perms:
            shuffled_blocks = [random.sample(b, len(b)) for b in perm]
            combined_order = list(itertools.chain.from_iterable(shuffled_blocks))
            trial_orders.append(combined_order)

    return trial_orders


def main_block():
    # 1. 保存管理クラスの初期化
    saver = SafeDriveSaver(OUTPUT_PATH)

    # 2. 再開位置の特定 (ローカルの一時ファイルに基づいてカウント)
    done_count = saver.get_current_count()
    print(f"Resume check: {done_count} lines already processed.")

    # モデル準備
    model, tokenizer = get_model_safe(MODEL_NAME)
    model.config.current_scale_map = None
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    # 3. ファイルオープン (saver.temp_file_path = ローカルパス を開く)
    #    ※ xopenは入力(gzip)には便利ですが、追記出力は標準openの方が安全かつ高速です
    with xopen(NQ_path, "r") as fin, open(saver.temp_file_path, "a", encoding="utf-8") as fout:

        # tqdmの設定 (done_count分は考慮済みとして表示させたい場合、initial引数を使うと便利です)
        pbar = tqdm(fin, total=DATA_NUM-VALIDATION_NUM)

        # セッション内での処理数カウント
        processed_in_session = 0

        for i, line in enumerate(pbar):
            # 再開スキップロジック
            # (done_count は既に保存済みの行数なので、それ以前は飛ばす)
            if i < done_count + VALIDATION_NUM:
                continue
            if i >= DATA_NUM:
                break

            # シード設定
            random.seed(i)

            try:
                data = json.loads(line)
                question = data["question"]

                documents = []
                for ctx in deepcopy(data["ctxs"]):
                    documents.append(Document.from_dict(ctx))
                if not documents:
                    # raise ValueErrorだと実験が止まるので、continue推奨ですが、元のままにします
                    raise ValueError(f"Did not find any documents: {data}")

                bm25_indices = get_bm25_indices(question, documents)
                bm25_reordered_documents = [documents[idx] for idx in bm25_indices]

                all_attnmaps = []
                trial_orders_pool = generate_block_permuted_orders(len(documents), REPEAT_NUM)

                # バッチ処理ループ
                for start_idx in range(0, SAMPLE_NUM, BATCH_SIZE):
                    end_idx = min(start_idx + BATCH_SIZE, SAMPLE_NUM)
                    trial_orders = trial_orders_pool[start_idx:end_idx]

                    attnmaps = get_attnmap(
                        question,
                        bm25_reordered_documents,
                        model, tokenizer, trial_orders
                    )

                    for trial_attnmap in attnmaps:
                        restored_attnmap = [None] * len(documents)
                        for j, score in enumerate(trial_attnmap):
                            original_idx = bm25_indices[j]
                            restored_attnmap[original_idx] = score.tolist()
                        all_attnmaps.append(restored_attnmap)

                    del attnmaps
                    torch.cuda.empty_cache()

                # 出力データ作成
                output_dict = {
                    "id": i,
                    "question": question,
                    "attnmap": all_attnmaps,
                }

                # --- 【ここが変更点】 ---
                # 1. ローカル一時ファイルへ保存 (明示的)
                saver.save_to_local(fout, output_dict)

                # 2. 定期的にDriveへ同期 (明示的)
                processed_in_session += 1
                if processed_in_session % SYNC_INTERVAL == 0:
                    pbar.set_description("Syncing to Drive...")
                    saver.sync_to_drive()
                    pbar.set_description("Processing")
                # -----------------------

            except Exception as e:
                print(f"Error at index {i}: {e}")
                # エラー時も念のため保存
                saver.sync_to_drive()
                raise e

    # 終了時の最終同期
    print("Finalizing save...")
    saver.sync_to_drive()

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
import random
random.seed(0)

NQ_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_block_random_6trial_llama.jsonl"


REPEAT_NUM = 1 # trial数は6*repeat_num
SAMPLE_NUM = 6 * REPEAT_NUM
BATCH_SIZE = 6

VALIATION_NUM = 600
DATA_NUM = 2655

MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"


if __name__ == "__main__":
  main_block()

NameError: name 'main_block' is not defined

In [ ]:
import random
random.seed(0)

NQ_path = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_block_random_12trial_llama.jsonl"


REPEAT_NUM = 2 # trial数は6*repeat_num
SAMPLE_NUM = 6 * REPEAT_NUM
BATCH_SIZE = 6

VALIATION_NUM = 600
DATA_NUM = 2655

MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"


if __name__ == "__main__":
  main_block()

✅ Model already loaded. Using cached model.


2655it [00:00, 27012.04it/s]


In [ ]:
import random
random.seed(0)

NQ_path = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
OUTPUT_PATH = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_block_random_12trial_vicuna-16k.jsonl"


REPEAT_NUM = 2 # trial数は6*repeat_num
SAMPLE_NUM = 6 * REPEAT_NUM
BATCH_SIZE = 3

VALIDATION_NUM = 600
DATA_NUM = 2655

MODEL_NAME = "lmsys/vicuna-7b-v1.5-16k"

if __name__ == "__main__":
  main_block()

🔄 Driveから既存データをロード中: /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_block_random_12trial_vicuna-16k.jsonl
Resume check: 2055 lines already processed.
✅ Model already loaded. Using cached model.


2655it [00:00, 24138.79it/s]            

Finalizing save...


In [6]:
import os
import json
import shutil
from xopen import xopen

def patch_specific_line(file_path: str, target_line_num: int, fix_callback):
    """
    指定した行番号のデータを修正して上書き保存する関数

    Args:
        file_path (str): Drive上のファイルパス
        target_line_num (int): 修正したい行番号 (1から始まる番号。例: 1322)
        fix_callback (func): 辞書データを受け取り、修正後の辞書を返す関数
    """
    print(f"🔧 Repairing line {target_line_num} in {os.path.basename(file_path)}...")

    # 作業用の一時ファイルパス
    temp_path = os.path.join("/content", "repair_temp.jsonl")

    # 修正カウンター
    patched = False

    try:
        # 読み込み(fin) と 書き込み(fout) を同時に開く
        with xopen(file_path, "r") as fin, open(temp_path, "w", encoding="utf-8") as fout:

            for i, line in enumerate(fin):
                # enumerateは0始まりなので、+1して行番号に合わせる
                current_line_num = i + 1

                if current_line_num == target_line_num:
                    # --- 修正対象の行 ---
                    try:
                        data = json.loads(line)
                        # コールバック関数でデータを修正
                        new_data = fix_callback(data)

                        # 書き込み
                        fout.write(json.dumps(new_data) + "\n")
                        print(f"  ✅ Line {current_line_num} fixed and written.")
                        patched = True
                    except Exception as e:
                        print(f"  ❌ Error processing line {current_line_num}: {e}")
                        raise e
                else:
                    # --- その他の行（そのままコピー） ---
                    fout.write(line)

        if not patched:
            print(f"⚠️ Warning: Target line {target_line_num} was not found (file implies shorter?).")
            return

        # --- Driveへの書き戻し ---
        print("💾 Backing up and overwriting original file...")

        # 1. バックアップ作成 (.bak)
        backup_path = file_path + ".bak_repair"
        shutil.copy2(file_path, backup_path)

        # 2. 修正版で上書き
        shutil.move(temp_path, file_path)
        print("✨ Repair Complete.")

    except Exception as e:
        print(f"🔥 Repair Failed: {e}")
        if os.path.exists(temp_path):
            os.remove(temp_path)

import json
import random
import torch
from xopen import xopen
from copy import deepcopy
import gc

# 1322行目のデータを生成する専用関数
def generate_single_line_data(target_index=1322):
    print(f"🚀 Generating data for line {target_index}...")

    # 1. モデルの準備 (ロード済みであればスキップされますが、念のため記述)
    # ※ ノートブック内で get_model_safe が定義されている前提です
    print("--- Loading Model ---")
    model, tokenizer = get_model_safe(MODEL_NAME)
    model.config.current_scale_map = None
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    # メモリ掃除
    gc.collect()
    torch.cuda.empty_cache()

    target_data = None

    # 2. 対象の行（1322行目）を探して読み込む
    print(f"--- Searching for line {target_index} in {NQ_path} ---")
    with xopen(NQ_path, "r") as fin:
        for i, line in enumerate(fin):
            if i != target_index:
                continue

            # === 対象行に到達 ===
            print(f"✅ Line {i} found. Processing...")

            # ★重要: 元のコードと同じシード値を設定して再現性を確保
            random.seed(i)

            # データ読み込み
            data = json.loads(line)
            question = data["question"]

            documents = []
            for ctx in deepcopy(data["ctxs"]):
                documents.append(Document.from_dict(ctx))

            if not documents:
                raise ValueError(f"Did not find any documents: {data}")

            # BM25ソート
            bm25_indices = get_bm25_indices(question, documents)
            bm25_reordered_documents = [documents[idx] for idx in bm25_indices]

            all_attnmaps = []

            # ブロック置換順序の生成 (generate_block_permuted_orders 関数が必要)
            # REPEAT_NUMなどが未定義の場合は、元の設定値(恐らく12 trialなので repeat=2?)を入れてください
            # ここでは元のコードの変数を参照します
            trial_orders_pool = generate_block_permuted_orders(len(documents), REPEAT_NUM)

            print(f"  - Inference started (Batch Size: {BATCH_SIZE}, Total Trials: {SAMPLE_NUM})...")

            # バッチ処理ループ
            for start_idx in range(0, SAMPLE_NUM, BATCH_SIZE):
                end_idx = min(start_idx + BATCH_SIZE, SAMPLE_NUM)
                trial_orders = trial_orders_pool[start_idx:end_idx]

                # 推論実行
                attnmaps = get_attnmap(
                    question,
                    bm25_reordered_documents,
                    model, tokenizer, trial_orders
                )

                # 順序の復元 (Restoration)
                for trial_attnmap in attnmaps:
                    restored_attnmap = [None] * len(documents)
                    for j, score in enumerate(trial_attnmap):
                        original_idx = bm25_indices[j]
                        restored_attnmap[original_idx] = score.tolist()
                    all_attnmaps.append(restored_attnmap)

                del attnmaps
                torch.cuda.empty_cache()

            # 出力データ作成
            target_data = {
                "id": i,
                "question": question,
                "attnmap": all_attnmaps,
            }
            break # 処理が終わったらループを抜ける

    if target_data is None:
        print(f"❌ Error: Line {target_index} not found in the file.")
        return None

    print(f"✨ Generation Complete for ID: {target_data['id']}")
    return target_data

# ==========================================
# 実行
# ==========================================

NQ_path = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
OUTPUT_PATH = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_block_random_12trial_vicuna-16k.jsonl"


REPEAT_NUM = 2 # trial数は6*repeat_num
SAMPLE_NUM = 6 * REPEAT_NUM
BATCH_SIZE = 3

VALIDATION_NUM = 600
DATA_NUM = 2655

MODEL_NAME = "lmsys/vicuna-7b-v1.5-16k"

START_IDX = 2534
END_IDX = 2536

generated_results = {}

print(f"🚀 Starting bulk generation from ID {START_IDX} to {END_IDX}...")

for target_idx in range(START_IDX, END_IDX + 1):
    data = generate_single_line_data(target_idx)
    if data:
        generated_results[target_idx + 1] = data  # 行番号(1始まり)をキーにして保存

    # 進行状況の保存（念のためローカルに追記）
    with open("/content/replace.jsonl", "a") as f:
        f.write(json.dumps(data) + "\n")

print(f"✅ Successfully generated {len(generated_results)} lines.")


🚀 Starting bulk generation from ID 2534 to 2536...
🚀 Generating data for line 2534...
--- Loading Model ---
⏳ Loading new model... (This may take a while)


config.json:   0%|          | 0.00/692 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/750 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✂️ Truncating model layers from 32 to 20
🔄 Swapping Attention Modules to DynamicScalingLlamaAttention...
✅ Model loaded and patched successfully.
--- Searching for line 2534 in /content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz ---
✅ Line 2534 found. Processing...
  - Inference started (Batch Size: 3, Total Trials: 12)...


/tmp/ipython-input-1271202468.py:132: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  offsets = torch.tensor(offset_mapping[batch_count])


✨ Generation Complete for ID: 2534
🚀 Generating data for line 2535...
--- Loading Model ---
✅ Model already loaded. Using cached model.
--- Searching for line 2535 in /content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz ---
✅ Line 2535 found. Processing...
  - Inference started (Batch Size: 3, Total Trials: 12)...
✨ Generation Complete for ID: 2535
🚀 Generating data for line 2536...
--- Loading Model ---
✅ Model already loaded. Using cached model.
--- Searching for line 2536 in /content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz ---
✅ Line 2536 found. Processing...
  - Inference started (Batch Size: 3, Total Trials: 12)...
✨ Generation Complete for ID: 2536
✅ Successfully generated 3 lines.


In [ ]:
import random
random.seed(0)

NQ_path = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
OUTPUT_PATH = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_block_random_6trial_vicuna-16k.jsonl"


REPEAT_NUM = 1 # trial数は6*repeat_num
SAMPLE_NUM = 6 * REPEAT_NUM
BATCH_SIZE = 3

VALIDATION_NUM = 600
DATA_NUM = 2655

MODEL_NAME = "lmsys/vicuna-7b-v1.5-16k"


if __name__ == "__main__":
  main_block()

In [ ]:
import time
import gc
import torch
from google.colab import runtime

# 1. メモリの最終開放（念のため）
gc.collect()
torch.cuda.empty_cache()

# 2. 待機設定
# Google Driveの同期には30〜60秒ほど見ておくと非常に安全です
WAIT_TIME_SEC = 300

print(f"✅ 全ての処理が完了しました。")
print(f"📂 Google Driveへの反映を待機しています（{WAIT_TIME_SEC}秒）...")

# プログレスバー付きで待機（あとどれくらいで消えるか視覚化）
from tqdm import tqdm
for _ in tqdm(range(WAIT_TIME_SEC)):
    time.sleep(1)

print("\n🚀 待機完了。セッションを終了し、ランタイムを削除します。お疲れ様でした！")

# 3. セッション終了
runtime.unassign()